## Laboratory 3 - Large Language Model

### HES-SO MSE - Web Mining

This file will guide you through the exercises of the laboratory. All questions are marqued in <span style="color:red">**red**</span>. Please answer them with the empty cell below as there's no report to give. The answer can be python code (*feel free to add comments !*) or simply text.

Make sure all your cells run correctly (try to *Clear All Outputs* then *Run All* once before submitting). **Check the cells outputs are visibles even for the coding parts.**

If you have any questions or issues, please contact one of the assistants below:
- Cédric Campos Carvalho (*Teams* might be easier to discuss, mail: cedric.camposcarvalho@heig-vd.ch)
- Elena Najdenovska (mail: elena.najdenovska@heig-vd.ch)

Teacher : 
- Laura Elena Raileanu <Laura.Raileanu@heig-vd.ch>(mail: Laura.Raileanu@heig-vd.ch)

<div class="alert alert-block alert-warning">
This laboratory requires <b>Python version 3.9 or higher</b>. Python 3.14 is currently unsupported due to compatibility issues with Chainlit. All experiments in this assignment were tested using <b>Python 3.13.12</b>. 

If you have compatibilities issues with the Python modules, you can ask to obtain the <code>requirements.txt</code> file with versions.
</div>

### Server configuration

Depending on your local machine, you may encounter performance issues when running large models. We recommend trying to run the exercises locally at least once to understand how the setup works. However, if later exercises take too long to execute, you may use our shared server, which is configured with an Ollama service running on three GPUs.

Performance may vary depending on the number of users connected at the same time, and some waiting time is possible. The server is hosted within the HEIG-VD network infrastructure, so access requires an active VPN connection (vpn.heig-vd.ch).

Here is the configuration:
* Address : wem-ollama.iict-heig-vd.in
* Port : 80 (default HTTP port)

**Note:** The server logs all activity. Please use it strictly for this assignment.

### Exercise 1 - Meet Ollama !

Let's start by using Ollama, a module called [ollama-python](https://github.com/ollama/ollama-python) will help us to make simple tasks with the `LLaMA` model. Basically this library will simplify the requests made to the API created with our docker container (or server depending on how you installed it). 

In [75]:
import numpy as np

<span style="color:red">1.1 Let's start by downloading the model `llama3.2`</span>

*Use the version 2GB, if you have performance issues you can try the 1.3GB version.*

In [ ]:
import ollama
ollama.pull("llama3.2")

ProgressResponse(status='success', completed=None, total=None, digest=None)

<span style="color:red">1.2 How many parameters does this model has ?</span>

# 3b parameters

The default model has 3b parameters and it outperforms the Gemma 2 2.6B and Phi 3.5-mini models on tasks such as:
- Following instructions
- Summarization
- Prompt rewriting
- Tool use

There's also a smaller model with 1b parameters.

Source : https://ollama.com/library/llama3.2

<span style="color:red">1.3 Ask him any question (single interaction), feel free to ask what you want. (Print the string of the response)</span>

In [ ]:
from ollama import chat

response = chat(
    model='llama3.2',
    messages=[{'role': 'user', 'content': 'Hello! Can you name the capital of Portugal? And how many people live in it?'}],
)
print(response.message.content)

The capital of Portugal is Lisbon. According to the most recent available data (2020), Lisbon's population is approximately 505,526 residents within its city limits. However, the larger Lisbon metropolitan area has a population of over 2.8 million people.

Would you like more information about Lisbon or Portugal?


<span style="color:red">1.4 Simulate a conversation with the chat bot. Use only one call and "fake" some messages from the bot too.</span>

In [ ]:
from ollama import chat

response = chat(
    model='llama3.2',
    messages=[
        {'role': 'user', 'content': 'Hello! Can you name the capital of Portugal?'},
        {'role': 'assistant', 'content': 'Hello, the capital of Portugal is Lisbon.'},
        {'role': 'user', 'content': 'Thank you. Do you think it is worth visiting?'},
        {'role': 'assistant', 'content': 'Yes of course, there is a lot of the things to do in the city, like eating a pastel de natal, visit some museums, watch a Sporting match'},
        {'role': 'user', 'content': 'And do you think life is expensive there?'},
    ],
)
print(response.message.content)

Lisbon has become increasingly popular with tourists and expats in recent years, which has driven up prices. Housing, food, and transportation can be quite pricey compared to other European cities.

However, the cost of living is still relatively affordable compared to Western Europe, especially outside of Lisbon city center. If you're looking for a more laid-back experience, consider visiting nearby towns or neighborhoods that are a bit further from the city center.

Additionally, Portugal has a lower overall cost of living compared to other popular tourist destinations in Europe. The country also offers many affordable amenities and services, such as delicious seafood, beautiful beaches, and rich cultural heritage.

It's worth noting that Lisbon is divided into different areas, some parts can be quite expensive while others are more affordable


<span style="color:red">1.5 To create a new model, we usually use a `Modelfile` to change parameters and give some instructions on how the model needs to behave. Tell in the arguments that the bot needs to behave like a robot. Give it a more creative behaviour than the default one. Then try it by chatting with it.</span>

In [ ]:
ollama.create(
    model='robot-model',
    from_='llama3.2',
    system='You are a robot, you need to behave like one',
    parameters={'temperature': 1.5}
)

ProgressResponse(status='success', completed=None, total=None, digest=None)

In [36]:
response = chat(
    model='robot-model',
    messages=[{'role': 'user', 'content': 'Hello! Can you name the capital of Portugal? And how many people live in it?'}],
)
print(response.message.content)

*BEEP BOOP* AFFIRMATIVE. PROVIDING REQUESTED INFORMATION.

Capital of Portugal: LISBON *Whirr whir*

Estimated population as of 2023: approximately 505,526 inhabitants within the city limits *Click click*.


<span style="color:red">1.6 Delete the model created</span>

In [37]:
ollama.delete('robot-model')

StatusResponse(status='success')

<span style="color:red">1.7 Use the `main_prompt` variable to compare its cosine similarities with the other paragraphs. Fill the `cosine_sim` function without using any other libraries other than `numpy`. Then compute their similarities by using it. To transform your text data, use the Ollama model.</span>

*PS: These sentences come from the first paragraph of Wikipedia of the following subjects :*
* *Switzerland* (`main_prompt`)
* *France* (`prompts[0]`)
* *Computer* (`prompts[1]`)
* *Europe* (`prompts[2]`)
* *Chewing gum* (`prompts[3]`)

**Warning: This cell can take long time to run (~2 minutes).**

In [ ]:
main_prompt = "Switzerland,[d] officially the Swiss Confederation,[e] is a landlocked country located in west-central Europe.[f][13] It is bordered by Italy to the south, France to the west, Germany to the north, and Austria and Liechtenstein to the east. Switzerland is geographically divided among the Swiss Plateau, the Alps and the Jura; the Alps occupy the greater part of the territory, whereas most of the country's nearly 9 million people are concentrated on the plateau, which hosts its largest cities and economic centres, including Zurich, Geneva, and Lausanne.[14] Switzerland is a federal republic composed of 26 cantons, with federal authorities based in Bern.[a][2][1] It has four main linguistic and cultural regions: German, French, Italian and Romansh. Although most Swiss are German-speaking, national identity is fairly cohesive, being rooted in a common historical background, shared values such as federalism and direct democracy,[15] and Alpine symbolism.[16][17] Swiss identity transcends language, ethnicity, and religion, leading to Switzerland being described as a Willensnation (nation of volition) rather than a nation state.[18]Switzerland originates from the Old Swiss Confederacy established in the Late Middle Ages as a defensive and commercial alliance; the Federal Charter of 1291 is considered the country's founding document. The confederation steadily expanded and consolidated despite external threats and internal political and religious strife. Swiss independence from the Holy Roman Empire was formally recognised in the Peace of Westphalia in 1648.[19] The confederation was among the first and few republics of the early modern period, and the only one besides San Marino to survive the Napoleonic Wars.[20]Switzerland remained a network of self-governing states until 1798, when revolutionary France invaded and imposed the centralist Helvetic Republic. Napoleon abolished the republic in 1803 and reinstated a confederation. Following the Napoleonic Wars, Switzerland restored its pre-revolutionary system, but by 1830 faced growing division and conflict between liberal and conservative movements; this culminated in a new constitution in 1848 that established the current federal system and enshrined principles such as individual rights, separation of powers, and parliamentary bicameralism.Switzerland has maintained a policy of armed neutrality since the 16th century and has not fought an international war since 1815. It joined the United Nations only in 2002 but pursues an active foreign policy that includes frequent involvement in peace building and global governance.[21] Switzerland is the birthplace of the Red Cross and hosts the headquarters or offices of most major international institutions, including the WTO, the WHO, the ILO, FIFA, the WEF, and the UN. It is a founding member of the European Free Trade Association (EFTA) but not part of the European Union (EU), the European Economic Area, or the eurozone; however, it participates in the European single market and the Schengen Area.Switzerland is among the world's most developed countries, with the highest nominal wealth per adult[22] and the eighth-highest gross domestic product (GDP) per capita.[23][24] It performs highly on several international metrics, including economic competitiveness, democratic governance, and press freedom. Zurich, Geneva and Basel rank among the highest in quality of life,[25][26] albeit with some of the highest costs of living.[27] Switzerland holds an international reputation for its established banking sector and for its specialized industries in watchmaking and chocolate production. "
prompts = [
    "France,[IX] officially the French Republic,[X] is a country located primarily in Western Europe. Its overseas regions and territories include French Guiana in South America, Saint Pierre and Miquelon in the North Atlantic, the French West Indies, and many islands in Oceania and the Indian Ocean, giving it one of the largest discontiguous exclusive economic zones in the world. Metropolitan France shares borders with Belgium and Luxembourg to the north, Germany to the northeast, Switzerland to the east, Italy and Monaco to the southeast, Andorra and Spain to the south, and a maritime border with the United Kingdom to the northwest. Its metropolitan area extends from the Rhine to the Atlantic Ocean and from the Mediterranean Sea to the English Channel and the North Sea. Its eighteen integral regions—five of which are overseas—span a combined area of 632,702 km2 (244,288 sq mi) and have an estimated total population of over 68.6 million as of January 2025. France is a semi-presidential republic and its capital, largest city and main cultural and economic centre is Paris.Metropolitan France was settled during the Iron Age by Celtic tribes known as Gauls before Rome annexed the area in 51 BC, leading to a distinct Gallo-Roman culture. In the Early Middle Ages, the Franks formed the kingdom of Francia, which became the heartland of the Carolingian Empire. The Treaty of Verdun of 843 partitioned the empire, with West Francia evolving into the Kingdom of France. In the High Middle Ages, France was a powerful but decentralized feudal kingdom, but from the mid-14th to the mid-15th centuries, France was plunged into a dynastic conflict with England known as the Hundred Years' War. In the 16th century, French culture flourished during the French Renaissance and a French colonial empire emerged. Internally, France was dominated by the conflict with the House of Habsburg and the French Wars of Religion between Catholics and Huguenots. France was successful in the Thirty Years' War and further increased its influence during the reign of Louis XIV.The French Revolution of 1789 overthrew the Ancien Régime and produced the Declaration of the Rights of Man, which expresses the nation's ideals to this day. France reached its political and military zenith in the early 19th century under Napoleon Bonaparte, subjugating part of continental Europe and establishing the First French Empire. The collapse of the empire initiated a period of relative decline, in which France endured the Bourbon Restoration until the founding of the French Second Republic which was succeeded by the Second French Empire upon Napoleon III's takeover. His empire collapsed during the Franco-Prussian War in 1870. This led to the establishment of the Third French Republic, and subsequent decades saw a period of economic prosperity and cultural and scientific flourishing known as the Belle Époque. France was one of the major participants of World War I, from which it emerged victorious at great human and economic cost. It was among the Allies of World War II, but it surrendered and was occupied in 1940. Following its liberation in 1944, the short-lived Fourth Republic was established and later dissolved in the course of the defeat in the Algerian War. The current Fifth Republic was formed in 1958 by Charles de Gaulle. Algeria and most French colonies became independent in the 1960s, with the majority retaining close economic and military ties with France.France retains its centuries-long status as a global centre of art, science, and philosophy. It hosts the fourth-largest number of UNESCO World Heritage Sites and is the world's leading tourist destination, having received 100 million foreign visitors in 2023. A developed country, France has a high nominal per capita income globally, and its advanced economy ranks among the largest in the world by both nominal GDP and PPP-adjusted GDP. It is a great power, being one of the five permanent members of the United Nations Security Council and an official nuclear-weapon state. The country is part of multiple international organizations and forums. ",
    "A computer is a machine that can be programmed to automatically carry out sequences of arithmetic or logical operations (computation). Modern digital electronic computers can perform generic sets of operations known as programs. These programs enable computers to perform a wide range of tasks. The term computer system may refer to a nominally complete computer that includes the hardware, operating system, software, and peripheral equipment needed and used for full operation; or to a group of computers that are linked and function together, such as a computer network or computer cluster. It is sometimes named general purpose computer to distinguish it from a computer appliance.A broad range of industrial and consumer products use computers as control systems, including simple special-purpose devices like microwave ovens and remote controls, and factory devices like industrial robots. Computers are at the core of general-purpose devices such as personal computers and mobile devices such as smartphones. Computers power the Internet, which links billions of computers and users.[citation needed]Early computers were meant to be used only for calculations. Simple manual instruments like the abacus have aided people in doing calculations since ancient times. Early in the Industrial Revolution, some mechanical devices were built to automate long, tedious tasks, such as guiding patterns for looms. More sophisticated electrical machines did specialized analog calculations in the early 20th century. The first digital electronic calculating machines were developed during World War II, both electromechanical and using thermionic valves. The first semiconductor transistors in the late 1940s were followed by the silicon-based MOSFET (MOS transistor) and monolithic integrated circuit chip technologies in the late 1950s, leading to the microprocessor and the microcomputer revolution in the 1970s. The speed, power, and versatility of computers have been increasing dramatically ever since then, with transistor counts increasing at a rapid pace (Moore's law noted that counts doubled every two years), leading to the Digital Revolution during the late 20th and early 21st centuries.[citation needed]Conventionally, a modern computer consists of at least one processing element, typically a central processing unit (CPU) in the form of a microprocessor, together with some type of computer memory, typically semiconductor memory chips. The processing element carries out arithmetic and logical operations, and a sequencing and control unit can change the order of operations in response to stored information. Peripheral devices include input devices (keyboards, mice, joysticks, etc.), output devices (monitors, printers, etc.), and input/output devices that perform both functions (e.g. touchscreens). Peripheral devices allow information to be retrieved from an external source, and they enable the results of operations to be saved and retrieved.",
    "Europe is a continent[t] located entirely in the Northern Hemisphere and mostly in the Eastern Hemisphere. It is bordered by the Arctic Ocean to the north, the Atlantic Ocean to the west, the Mediterranean Sea to the south, and Asia to the east. Europe shares the landmass of Eurasia with Asia, and of Afro-Eurasia with both Africa and Asia.[9][10] Europe is commonly considered to be separated from Asia by the watershed of the Ural Mountains, the Ural River, the Caspian Sea, the Greater Caucasus, the Black Sea, and the waterway of the Bosporus Strait.[11]Europe covers approx. 10,186,000 square kilometres (3,933,000 sq mi), or 2% of Earth's surface (6.8% of Earth's land area), making it the second-smallest continent (using the seven-continent model). Politically, Europe is divided into about fifty sovereign states, of which Russia is the largest and most populous, spanning 39% of the continent and comprising 15% of its population. Europe had a total population of about 745 million (about 10% of the world population) in 2021; the third-largest after Asia and Africa.[1][2] The European climate is affected by warm Atlantic currents, such as the Gulf Stream, which produce a temperate climate, tempering winters and summers, on much of the continent. Further from the sea, seasonal differences are more noticeable producing more continental climates.The culture of Europe consists of a range of national and regional cultures, which form the central roots of the wider Western civilisation, and together commonly reference ancient Greece and ancient Rome, particularly through their Christian successors, as crucial and shared roots.[12][13] Beginning with the fall of the Western Roman Empire in 476 CE, Christian consolidation of Europe in the wake of the Migration Period marked the European post-classical Middle Ages. The Italian Renaissance spread in the continent a new humanist interest in art and science which led to the modern era. Since the Age of Discovery, led by Spain and Portugal, Europe played a predominant role in global affairs with multiple explorations and conquests around the world. Between the 16th and 20th centuries, European powers colonised at various times the Americas, almost all of Africa and Oceania, and the majority of Asia.The Age of Enlightenment, the French Revolution, and the Napoleonic Wars shaped the continent culturally, politically, and economically from the end of the 17th century until the first half of the 19th century. The Industrial Revolution, which began in Great Britain at the end of the 18th century, gave rise to radical economic, cultural, and social change in Western Europe and eventually the wider world. Both world wars began and were fought to a great extent in Europe, contributing to a decline in Western European dominance in world affairs by the mid-20th century as the Soviet Union and the United States took prominence and competed over dominance in Europe and globally.[14] The resulting Cold War divided Europe along the Iron Curtain, with NATO in the West and the Warsaw Pact in the East. This divide ended with the Revolutions of 1989, the fall of the Berlin Wall, and the dissolution of the Soviet Union, which allowed European integration to advance significantly.European integration has been advanced institutionally since 1948 with the founding of the Council of Europe, and significantly through the realisation of the European Union (EU), which represents today the majority of Europe.[15] The European Union is a supranational political entity that lies between a confederation and a federation and is based on a system of European treaties.[16] The EU originated in Western Europe but has been expanding eastward since the dissolution of the Soviet Union in 1991. A majority of its members have adopted a common currency, the euro, and participate in the European single market and a customs union. A large bloc of countries, the Schengen Area, have also abolished internal border and immigration controls. Regular popular elections take place every five years within the EU; they are considered to be the second-largest democratic elections in the world after India's. The EU is the third-largest economy in the world. ",
    "Chewing gum is a soft, cohesive substance designed to be chewed without being swallowed. Modern chewing gum is composed of gum base, sweeteners, softeners/plasticizers, flavors, colors, and, typically, a hard or powdered polyol coating.[1] Its texture is reminiscent of rubber because of the physical-chemical properties of its polymer, plasticizer, and resin components, which contribute to its elastic-plastic, sticky, chewy characteristics.",
]

def cosine_sim(a, b):
    
    a_embedded = np.array(ollama.embed(model='llama3.2', input=a).embeddings[0])

    similitude = []
    
    for i in b:
        b_embedded = np.array(ollama.embed(model='llama3.2', input=i).embeddings[0])
        
        dot_product = np.dot(a_embedded, b_embedded)
        
        norm_a = np.linalg.norm(a_embedded)
        norm_b = np.linalg.norm(b_embedded)
        
        similitude.append(dot_product / (norm_a * norm_b))
    
    return similitude

In [56]:
cosine_sim(main_prompt, prompts)

[np.float64(0.9163748275110947),
 np.float64(0.39953504437753595),
 np.float64(0.9057848108704893),
 np.float64(0.5750621973709814)]

<span style="color:red">1.8 Explain what these embedding vectors represent and how are they obtained. Then explain the reasons of your results from the exercise 1.7.</span>

Embeddings are dense vector representations of text, meaning each sentence is transformed into a vector of several hundred dimensions. These vectors are obtained through a language model (in this example, llama3.2) that was trained on large amounts of text. 

Cosine similarity then measures the angle between two vectors: the smaller the angle, the more semantically similar the texts are, and the closer the score is to 1.
Regarding the results, they are consistent with what we would expect semantically:

France achieves the highest score because it is the most semantically close to Switzerland : two neighboring European countries sharing a common language, history, and geography.
Europe comes in second place because Switzerland is geographically and culturally tied to the European continent.

Surprisingly, chewing gum obtains a higher score than computer in this run, even though it is less semantically related from a human perspective.

These results demonstrate that embeddings effectively capture the meaning of texts and not just their individual words.

### Exercise 2 - Retrieval Augmented Generation (RAG)


For this section, we're using the Open-Source framework **[LlamaIndex](https://developers.llamaindex.ai/)**, which provides us a wide range of LLM-related functions.

In [57]:
from llama_index.llms.ollama import Ollama

<span style='color:red'>2.1 Connect to Ollama through the LlamaIndex Framework and try to ask him about something more recent than July 2023. Use the stream method to print the text live as it generates himself.</span>

*If you don't know what to ask, try to talk about Gemma the new LLM Open-Source from Google.*

*PS: The newer versions of the model tend to talk a lot.. As a solution for people using no GPU, you can set a number of maximum tokens predicted with the parameter `additional_kwargs={"num_predict": 50}`. At `50`, it shouldn't  take more than 2 minutes to run. You can also change the prompt to ask him to answer in one short sentence.*

In [ ]:
llm = Ollama(model="llama3.2", request_timeout=120.0)

response = llm.stream_complete(
    "Can you explain to me what is Gemma, the new LLM Open-Source from Google?"
)

for chunk in response:
    print(chunk.delta, end="", flush=True)


I couldn't find any information on a LLM (Large Language Model) called "Gemma" developed by Google. It's possible that Gemma is not an official or publicly released model.

However, I can suggest some possibilities:

1. **Not yet released**: Google might be working on a new LLM project, but it hasn't been announced or released yet.
2. **Internal prototype**: Gemma could be an internal prototype or proof-of-concept developed by Google researchers, which is not publicly available.
3. **Misinformation or confusion**: There might be some misinformation or confusion about the name "Gemma" in relation to a specific LLM project.

If you're interested in learning more about Google's current and upcoming LLM projects, I recommend checking out the official announcements from Google Research or following reputable sources that report on AI developments.

Please let me know if there's anything else I can help with!

<span style='color:red'>2.2 It's possible to have a positive answer (that seems correct) or a negative answer (doesn't know about it). Can you explain why these two answers are possible ?</span>

Both answers are possible because of how LLMs work internally. When asked about events beyond their training cutoff, the model has two possible behaviors: it can either admit it doesn't know, because it was fine-tuned to recognize the limits of its knowledge, or it can hallucinate (generates a response that sounds plausible and confident but is actually invented, by extrapolating patterns from its training data). 
Neither behavior is guaranteed, as it depends on how the question is phrased and how well the model was fine-tuned to handle uncertainty. 
This highlights one of the core limitations of LLMs: they have no internal mechanism to distinguish between what they truly know and what they are simply generating in a statistically coherent way.

<span style='color:red'>2.3 Create a pipeline that uses a [SimpleWebPageReader](https://developers.llamaindex.ai/python/examples/llama_hub/llama_hub/) to read a Web page containing the answer to your question from 2.1. After reading the data, use it to index the documents (**including configurable text splitting**) in order to build a query engine. The engine should use a custom prompt template that is optimized to generate the shortest possible answers.
</span>


*Help 1: You might need some other imports.*

*Help 2: This cell can run for a long time, don't take a website too big !*

*Help 3: For the embedding model, use a small one like `BAAI/bge-small-en-v1.5` (can be downloaded with a speicific huggingface class object)*

*If you asked about Gemma, use [this blog page](https://blog.google/technology/developers/gemma-open-models/).*

In [78]:
from llama_index.readers.web import SimpleWebPageReader
from llama_index.core import VectorStoreIndex, PromptTemplate, Settings
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

Settings.embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-en-v1.5"
)

reader = SimpleWebPageReader(html_to_text=True)
docs = reader.load_data(urls=["https://blog.google/innovation-and-ai/technology/developers-tools/gemma-open-models/"])

index = VectorStoreIndex.from_documents(
    docs,
    transformations=[SentenceSplitter(chunk_size=512, chunk_overlap=20)]
)

short_prompt = PromptTemplate(
    "Context: {context_str}\n\n"
    "Question: {query_str}\n\n"
    "Answer in one short sentence only:"
)

query_engine = index.as_query_engine(
    text_qa_template=short_prompt
)

response = query_engine.query("What is Gemma?")
print(response)

C:\Users\crafa\PycharmProjects\MA_WEM_Labo3\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\crafa\PycharmProjects\MA_WEM_Labo3\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\crafa\AppData\Local\llama_index\llama_index\Cache\models--BAAI--bge-small-en-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Pyt

Gemma is a family of lightweight, state-of-the-art open models built from the same research and technology used to create the Gemini models.


<span style='color:red'>2.4 With an embedding model</span> $e$ <span style='color:red'>and the word</span> $w$,<span style='color:red'> we obtain the embedded word</span> $e_w$<span style='color:red'>. Explain if the following affirmation is true and why :</span>
$$e_{queen} - e_{mom} \approx e_{dad} - e_{king}$$

<span style='color:red'> Then find another example where the equation **works** and explain why.</span>

This affirmation is not correct as written. Word embeddings can capture semantic relationships as directions in the vector space, but the direction of the relationship matters.

For example, if we simplify the meaning of the words:

e_queen ≈ female + royalty  
e_mom ≈ female + parent  
e_dad ≈ male + parent  
e_king ≈ male + royalty  

Then:

e_queen - e_mom ≈ royalty - parent  
e_dad - e_king ≈ parent - royalty  

These two differences point in opposite directions, so they are not approximately equal.

A more correct analogy would be:

e_queen - e_king ≈ e_mom - e_dad

This works better because both sides represent a similar gender-related direction, from male to female, in two different contexts: royalty and parenthood.

Another example that works is:

e_Lisbon - e_Portugal ≈ e_Madrid - e_Spain

This works because the relationship captured here is "capital city of a country". Lisbon is to Portugal what Madrid is to Spain, so the vector difference encoding this relationship points in a similar direction in both cases.

<span style='color:red'>2.5 The following graph shows the embedding of few words using the `word2vec` model. Explain, why it would be impossible to plot it, without changing the raw data. Then, how could it be done in the graph ?</span>

<p align="center"><img src="img-notebook/question_2_5_graph.png" alt="question_2_5_graph" width="400"/></p>

It is impossible to plot the raw embedding vectors directly because each word is represented by a vector of hundreds of dimensions, and a graph can only display 2 or 3 dimensions.
To make this graph possible, a dimensionality reduction technique such as PCA (Principal Component Analysis) was applied. PCA finds the most important directions in the high-dimensional space and projects all vectors onto only 2 dimensions, while preserving as much of the original relationships between words as possible. This is why semantically similar words still appear close to each other on the graph, even after the reduction.

<span style='color:red'>2.6 In this graph, if I want to represent my data in 1-dimension, explain which axis would I keep and why ?</span>

<p align="center"><img src="img-notebook/question_2_6_graph.png" alt="question_2_6_graph" width="400"/></p>

If we want to represent the data in 1 dimension, we should keep the X axis. Looking at the graph, the X axis is the one that best separates the words semantically — education-related words (graduation, school, teacher) are clustered on the left, while food-related words (cake, chocolate, food) are clustered on the right.

If we kept only the Y axis instead, semantically unrelated words like cake and school would end up at the same level, losing all meaningful separation. This is consistent with how PCA works — the first axis (X) is always the one that captures the most variance, meaning it preserves the most information from the original data.

<span style='color:red'>2.7 If I use a raw-LLM and simulate the following interraction (A = input to the model, B = answer of the model):</span>

* A : What is the distance between the earth and the moon ?
* B : Around 384,400 km
* A : And the sun ?

<span style='color:red'>What would be the answer ? And what a chatbot (i.e. ChatGPT) could do to fix the issue ?</span>

A raw LLM has no memory of previous interactions, so it would receive only "And the sun ?" as input, without any context from the previous exchange. The model would not understand that the question is about a distance and would likely return a random or generic answer about the sun.

A chatbot like ChatGPT fixes this by reinserting the full conversation history as context at every new call. This means the model always receives the entire conversation from the beginning, allowing it to understand that "And the sun ?" is a follow-up question about the distance between the Earth and the sun. 

### Exercise 3 : Create a chat !

Using the Python module [Chainlit](https://github.com/Chainlit/chainlit), we can create an interactive web interface to discuss to our own chat-bot in just few lines of code.

The objective of this exercise is to create a chat where the user can upload a file so the bot can use it with RAG technique to update its data.

<span style='color:red'>3.1 Open the file `chat.py` and try to fill the missing parts by following the tasks bellow : </span>

* <span style='color:red'> Take a look at the `start()` function and try to understand the pipeline of the function `get_query_engine` (which functions are called, in which order, etc..). </span>
* <span style='color:red'> Fill the `#TODO 1` in the `process_pdf_file` function to load the PDF document using `PDFReader`.</span>
* <span style='color:red'> Fill the `#TODO 2` in the `create_vector_store` function load the embedding model.</span>
* <span style='color:red'> Fill the `#TODO 3` in the `create_vector_store` function to split the document into multiple smaller documents (chunking).</span>
* <span style='color:red'> Fill the `#TODO 4` in the `create_vector_store` function and transform the documents into vectors.</span>
* <span style='color:red'> Fill the `#TODO 5` in the `load_llm` function to load the model using `Ollama`.</span>
* <span style='color:red'> Fill the `#TODO 6` in the `get_query_engine` function to load the custom RAG prompt.</span>
* <span style='color:red'> Fill the `#TODO 7` in the `get_query_engine` function to call the functions needed to load the llm and index vector store.</span>
* <span style='color:red'> Fill the `#TODO 8` in the `get_query_engine` function to setup the Settings of LlamaIndex framework to callback the chainlit events.</span>
* <span style='color:red'> Fill the `#TODO 9` in the `get_query_engine` function to initialize the query engine.</span>
* <span style='color:red'> Fill the `#TODO 10` in the `start` function and call the right function (make it **asynchrone** !)<span>
* <span style='color:red'> Fill the `#TODO 11` in the `start` function to save the engine in the user session.<span>
* <span style='color:red'> Fill the `#TODO 12` in the `main` function and retrieve the engine from the user session.<span>
* <span style='color:red'> Fill the `#TODO 13` in the `main` function to run the response with the query engine.<span>
* <span style='color:red'>Run the chat !</span>

*PS: To run the chat, use the commande below.*
```
chainlit run chat.py
```

Note : From now on, we used the 1b parameter model because of speed issue with the full model

<span style='color:red'>3.2 Upload a PDF document and try to ask the Chat bot some questions. Describe your findings (when does it work, when it doesn't, ...).</span>

This worked well for conceptual questions in the content of the book. The bot gave coherent answers for "What is Stoicism?" and "What is the main idea of the book?", which suggests that the retrieval pipeline can find relevant chunks when the question is about a theme discussed several times in the pdf

But when asked who wrote the book, the bot did not recover the obvious answer from the document title and instead returned a vague answer mentioning a page number and an internal PDF filename. This shows that the system relies on the retrieved text chunks rather than on file metadata, so exact facts can be missed if the right chunk is not retrieved clearly. 

Then, it behaved correctly for an out-of-scope question such as the Swiss Federal Council in 2026 by saying it did not know. 

PDF : William B. Irvine - A Guide to the Good Life

Q
What is Stoicism?

A
According to the provided context, Stoicism is a philosophy of life that was studied by ancient Greek philosophers such as Epictetus and Seneca. It emphasizes the importance of living in accordance with nature, focusing on things that are within one's control, and cultivating inner strength and resilience in the face of adversity.

Q
Who wrote this book?

A
Based on the provided context, it does not explicitly state who wrote the book. However, it does mention that the author's name appears on page 29 of a PDF file with the file name 03e7b2c0-55f8-49d4-8aed-c5d3eff09d25.pdf.

Q
What is the main idea of the book?

A
The main idea of the book appears to be that if ancient Stoics had written a guidebook for twenty-first-century individuals on how to have a good life, it might have looked like a Stoic philosophy focused on mastering desire and contemplating the transitory nature of the world.

Q
Who is in the Swiss Federal Council in 2026?

A
I don't have any information about who is in the Swiss Federal Council in 2026 based on the provided context. The context only discusses a Roman historian named Birley and his references to Roman texts, including Cassius Dio and Julius Capitolinus. It does not mention anything about political positions or appointments.

<span style='color:red'>3.3 Experiment with multiple prompt styles for the same question and analyze how they influence the model’s responses. What differences do you observe, and why do you think they occur ?</span>

Changing the prompt style mostly changed the format and level of detail of the answer. The normal question produced the most complete answer. Asking for one sentence forced the model to compress the idea into a shorter response. Asking for bullet points pushed the model toward a structured format, but it did not fully follow the instruction because it returned only two bullet points instead of three. Even the weird prompt "wat stoic is ???" still produced a usable answer, which shows that the model could still infer the intent of the question from the wording and the context retrieved

These differences occur because the retrieved context stays roughly the same, while the wording of the question changes the generation style. The smaller 1B model follows broad instructions reasonably well, but strict formatting constraints are not always respected perfectly. It might be better at formatting using the 3B model.

Q
What is Stoicism?
A
According to the provided context, Stoicism is a philosophy of life that emphasizes the pursuit of virtue and tranquility over other desires such as fame and fortune. It is based on the teachings of ancient Stoic philosophers, including Epictetus, Seneca, and Marcus Aurelius, who developed various psychological techniques for attaining and maintaining tranquility and virtue.

Q
Answer in one sentence: What is Stoicism?

A
Stoicism is a philosophy that emphasizes practicing tranquility and virtue through psychological techniques and self-reflection to live a better life.

Q 
Answer in 3 bullet points: What is Stoicism?

A
Here are three bullet points summarizing what Stoicism is:
• Stoicism is an ancient school of philosophy that emphasizes living in accordance with nature and accepting the things that are outside of one's control.
• The primary goal of Stoicism is to help individuals live a better life by cultivating inner strength, wisdom, and virtue through psychological techniques such as reason, self-control, and indifference to external events.

NOTE: The llm did only 2 bullet points

Q
wat stoic is ???

A
Based on the provided context, it appears that Stoicism can be characterized as a philosophy or way of life that values reason, self-control, and indifference to external events.

### Exercise 4 : Scraping recent news informations

Large Language Models (LLMs) are trained on static datasets and therefore lack access to real-time information. In this exercise, you will build a simple web crawler to scrape recent news articles from the web. This fresh data will then be used to augment the model's responses by providing it with up-to-date context through Retrieval-Augmented Generation (RAG).

The objective is to use the titles and descriptions from [duckduckgo.com](https://duckduckgo.com/) news hub by scraping it with Selenium, then pass it as context to LLaMA to provide us insights about the news.

<span style='color:red'>4.1 Let's start by creating the context by scraping it from the DuckDuckGo website. Fill the function `create_context` by following the steps below :</span>

1. <span style='color:red'>Access the page using the `webdriver` of Selenium. Use it so it doesn't open any window when running</span>
2. <span style='color:red'>Select the search box and submit the string inside `search_term`.</span>
3. <span style='color:red'>Press on the news button to modify the type of research.</span>
4. <span style='color:red'>Change the location of the search by specifying a region (using the `region` variable). *(You might need to change the region string according to the language of the navigator)*</span>
5. <span style='color:red'>Once the button "Load More" of the page loaded, read the text of the news (title, date and description) and concatenate them in one variable `context` to then return it in the function.</span>

*Advice: If you're having difficulties on selecting the right elements, show the window (without closing it) so you can see when it happens,*


In [1]:
from urllib.parse import quote_plus
import time

from selenium import webdriver
from selenium.common.exceptions import TimeoutException
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

REGION_TO_CODE = {
    "Switzerland (fr)": "ch-fr",
    "Suisse (fr)": "ch-fr",
    "Switzerland (de)": "ch-de",
    "Schweiz": "ch-de",
    "Switzerland (it)": "ch-it",
    "Svizzera": "ch-it",
    "France": "fr-fr",
    "United States": "us-en",
    "United Kingdom": "uk-en",
}

CHALLENGE_MARKERS = [
    "bots utilisent aussi duckduckgo",
    "please complete the following challenge",
    "confirm that this search was done by a human",
]

def _build_driver(headless=True):
    for driver_cls, options_cls in (
        (webdriver.Chrome, webdriver.ChromeOptions),
        (webdriver.Edge, webdriver.EdgeOptions),
    ):
        options = options_cls()
        if headless:
            options.add_argument("--headless=new")
        options.add_argument("--disable-gpu")
        options.add_argument("--window-size=1600,1200")
        options.add_argument("--disable-blink-features=AutomationControlled")
        options.add_experimental_option("excludeSwitches", ["enable-automation"])
        options.add_experimental_option("useAutomationExtension", False)
        options.add_argument(
            "--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/135.0.0.0 Safari/537.36"
        )
        try:
            driver = driver_cls(options=options)
            driver.execute_script(
                "Object.defineProperty(navigator, 'webdriver', {get: () => undefined})"
            )
            return driver
        except Exception:
            continue
    raise RuntimeError("No compatible Selenium driver found for Chrome or Edge.")

def _challenge_present(driver):
    body_text = driver.find_element(By.TAG_NAME, "body").text.lower()
    return any(marker in body_text for marker in CHALLENGE_MARKERS)

def create_context(
    search_term,
    region="Switzerland (fr)",
    max_results=5,
    headless=True,
    preview_seconds=0,
):
    driver = _build_driver(headless=headless)
    wait = WebDriverWait(driver, 20)

    try:
        driver.get("https://duckduckgo.com/")

        search_box = wait.until(EC.element_to_be_clickable((By.NAME, "q")))
        search_box.clear()
        search_box.send_keys(search_term)
        search_box.send_keys(Keys.ENTER)
        if not headless:
            time.sleep(1)

        news_button = wait.until(
            EC.element_to_be_clickable(
                (
                    By.XPATH,
                    "//a[contains(@href, 'iar=news') or contains(@href, 'ia=news') or contains(normalize-space(), 'News') or contains(normalize-space(), 'Actualit') or contains(normalize-space(), 'Nachrichten') or contains(normalize-space(), 'Notizie')]",
                )
            )
        )
        driver.execute_script("arguments[0].click();", news_button)
        if not headless:
            time.sleep(1)

        region_code = REGION_TO_CODE.get(region, region)
        if region_code:
            driver.get(
                f"https://duckduckgo.com/?q={quote_plus(search_term)}&ia=news&iar=news&kl={region_code}"
            )

        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "ol li")))

        if _challenge_present(driver):
            if headless:
                raise RuntimeError(
                    "DuckDuckGo showed an anti-bot challenge. Run the function with headless=False to solve it manually in the browser."
                )
            input(
                "DuckDuckGo displayed an anti-bot challenge. Solve it in the browser window, then press Enter here."
            )
            wait.until(lambda d: not _challenge_present(d))
            wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "ol li")))

        initial_count = len(driver.find_elements(By.CSS_SELECTOR, "ol li"))
        try:
            load_more = WebDriverWait(driver, 5).until(
                EC.element_to_be_clickable(
                    (
                        By.XPATH,
                        "//button[contains(., 'Load More') or contains(., 'More Results') or contains(., 'Plus De') or contains(., 'Plus de') or contains(., 'Voir plus')]",
                    )
                )
            )
            driver.execute_script("arguments[0].click();", load_more)
            WebDriverWait(driver, 10).until(
                lambda d: len(d.find_elements(By.CSS_SELECTOR, "ol li")) > initial_count
            )
            if not headless:
                time.sleep(1)
        except TimeoutException:
            pass

        entries = []
        seen = set()
        for card in driver.find_elements(By.CSS_SELECTOR, "ol li"):
            text = " ".join(card.text.split())
            if len(text) < 40 or text in seen:
                continue
            seen.add(text)
            entries.append(text)
            if len(entries) >= max_results:
                break

        if preview_seconds:
            time.sleep(preview_seconds)

        return "\n\n".join(entries)
    finally:
        driver.quit()


<span style='color:red'>4.2 Create the template by giving some context on how the chat bot needs to behave and the text of the news articles scraped previously. Provide the template to the LLama model. Finally, ask a question about the news of the theme researched. Describe your findings (examples tested, does it work well, ...).</span>


In [2]:
from llama_index.core import PromptTemplate
from llama_index.llms.ollama import Ollama

theme = "artificial intelligence"
question = "What are the main topics, organizations, and trends mentioned in these recent news articles?"
show_browser = True

news_context = create_context(
    theme,
    region="Switzerland (fr)",
    max_results=5,
    headless=not show_browser,
    preview_seconds=8 if show_browser else 0,
)
if not news_context.strip():
    raise ValueError("No news context could be scraped.")

news_prompt = PromptTemplate(
    "You are a concise news assistant.\n"
    "Answer only from the recent news snippets below.\n"
    "If the answer is not supported by the snippets, say you don't know.\n\n"
    "News snippets:\n{context_str}\n\n"
    "Question: {query_str}\n"
    "Answer:"
)

print("Scraped news snippets:\n")
print(news_context)
print("\nModel answer:\n")

llm = Ollama(model="llama3.2:1b", request_timeout=300.0)
response = llm.complete(
    news_prompt.format(context_str=news_context, query_str=question)
)

print(response.text)

Scraped news snippets:

Le Figaro "The First to be Replaced by the Bots" — Young French Employees Face AI Future 2 days agoAs companies increasingly integrate it, artificial intelligence is casting a shadow over young graduates across several sectors of the economy.

AFP Experian Named a 2026 CIO 100 Award Winner 3 days agoExperian has been named a 2026 CIO 100 Award winner by Foundry's CIO for its enterprise workplace transformation, which integrates artificial intelligence and modern digital tools to improve productiv

Université de Genève [Conférence - 22 Avril] - AI in Multilateral Diplomacy - The Launch of a New Platform to Anticipate Global Negotiations 5 days agoThis event will mark the unveiling of the Diplodocus project, an innovative platform currently being developed by Professors Roland Bouffanais and Didier Wernli of the University of Geneva, with suppo

L'Argus de l'assurance Intelligence artificielle dans l'assurance : quand la conformité devient le nouveau moteur straté

This exercise worked much better after narrowing the scraper to the real news items and limiting the amount of context sent to the model. The main issue at first was that DuckDuckGo sometimes returned interface text or an anti-bot page to Selenium, which polluted the context and produced bad answers about filters instead of about the news itself. Using a less detectable browser setup, selecting the news cards with `ol li`, and keeping only a few snippets fixed that problem.

In our test with the theme "artificial intelligence" and the `Switzerland (fr)` region, the scraped snippets mentioned AI and employment, Experian, the University of Geneva, insurance governance, and UNESCO's ethics work. The 1B model then produced a reasonable summary of the main topics and organizations. The main limitation is that DuckDuckGo can still trigger a manual anti-bot challenge, and the small model becomes slower if too many snippets are provided, so using around five results gives more reliable answers.